### Environment Check

In [ ]:
# This notebook requires: numpy, pandas, scipy, and statsmodels.
# If you are missing any, run: !pip install numpy pandas scipy statsmodels

import importlib
import sys

dependencies = ['numpy', 'pandas', 'scipy', 'statsmodels']
missing = [pkg for pkg in dependencies if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"⚠️ Missing libraries: {missing}")
    print(f"Please run: !{sys.executable} -m pip install {' '.join(missing)}")
else:
    print("✅ All dependencies found.")

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import t as scipy_t
import statsmodels.api as sm

In [ ]:
# Set seed for reproducibility
np.random.seed(42)

### 1. Path Configuration

In [ ]:
# Finds the root directory running from either the 'scripts' folder or the project root, ensuring correct path resolution for data files
root_dir = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()

raw_data_path = root_dir / "data" / "raw" / "onion_sampling_raw.csv"
processed_data_path = root_dir / "data" / "processed" / "onion_farm_layout_metric.csv"

### 2. Data Pre-processing


In [ ]:

# Remove existing processed data to ensure a clean slate for the new analysis
if os.path.exists(processed_data_path):
    os.remove(processed_data_path)
    print(f"Existing processed file '{processed_data_path}' removed.")

# Load raw data
raw_df = pd.read_csv(raw_data_path)

# Calculate derived metrics
raw_df['Effective_Bed_Area_m2'] = raw_df['Bed_Length_m'] * raw_df['Bed_Width_m']
raw_df['Canal_Area_m2'] = raw_df['Canal_Length_m'] * raw_df['Canal_Width_m']
raw_df['Seedling_Density_per_m2'] = raw_df['Seedling_Count'] / raw_df['Effective_Bed_Area_m2']

# Calculate Land-Use Efficiency (Ri)
raw_df['Ri'] = raw_df['Effective_Bed_Area_m2'] / (raw_df['Effective_Bed_Area_m2'] + raw_df['Canal_Area_m2'])

# Save to processed folder for the statistical models
os.makedirs(root_dir / ".."/ "data" / "processed", exist_ok=True)
raw_df.to_csv(processed_data_path, index=False)

print("Data successfully processed and saved.")

# Data Integrity Verification

EXPECTED_SHAPE = (50, 12)
EXPECTED_SEEDLING_SUM = 13966

def verify_integrity(df: pd.DataFrame):
    current_sum = df['Seedling_Count'].sum()
    if df.shape == EXPECTED_SHAPE and current_sum == EXPECTED_SEEDLING_SUM:
        print("✅ Data Integrity Verified: Dataset matches publication records.")
    else:
        print("⚠️ Warning: Dataset mismatch. Results may differ from publication.")
        print(f"Expected: {EXPECTED_SHAPE}, Sum: {EXPECTED_SEEDLING_SUM}")
        print(f"Found: {df.shape}, Sum: {current_sum}")

verify_integrity(raw_df)

### 3. Descriptive Statistics

In [ ]:
df_stats = df.describe()[["Bed_Length_m", "Bed_Width_m", "Effective_Bed_Area_m2", "Seedling_Count", "Seedling_Density_per_m2", "Ri"]]
print(df_stats)

mean_density = df["Seedling_Density_per_m2"].mean()
std_density = df["Seedling_Density_per_m2"].std(ddof=1)
cv_density = std_density / mean_density

print(f"\nMean Density: {mean_density:.2f} plants/m2")
print(f"Std Deviation: {std_density:.2f}")
print(f"CV: {cv_density:.2%}")

### 4. Confidence Interval (CLT-based)

In [ ]:
alpha = 0.05
n = len(df)
t_crit = scipy_t.ppf(1 - alpha/2, df=n-1)

# Margin of Error
moe = t_crit * (std_density / np.sqrt(n))

ci_lower = mean_density - moe
ci_upper = mean_density + moe

print(f"95% Confidence Interval for Mean Density: [{ci_lower:.2f}, {ci_upper:.2f}]")


### 5. Planning Framework (Application of Results)

In [ ]:
# Example: Estimate seedlings needed for 100 m² of bed area.
area_target = 100  # m^2
expected_seedlings = mean_density * area_target
lower_bound = ci_lower * area_target
upper_bound = ci_upper * area_target

print(f"\nFor {area_target}m2 area:")
print(f"Expected Seedlings: {expected_seedlings:.0f}")
print(f"95% Prediction Interval: [{lower_bound:.0f}, {upper_bound:.0f}]")

### 6. Regression Analysis: Ri vs Density

In [ ]:
# Investigating if land-use efficiency correlates with planting density
X = sm.add_constant(df["Ri"])
y = df["Seedling_Density_per_m2"]

model = sm.OLS(y, X).fit()
print(model.summary())